# 03 — Ten-day forecast initialised from the DA analysis

The end-to-end chain: ERA5 + observations → **cascade DA** (notebook 02) →
**10-day forecast from the analysis** (this notebook), scored against ERA5
and compared with the ERA5-initialised baseline of notebook 01.

**Prerequisites**: notebook 02 finished (needs
`../outputs/demo_dacycle/ic_6h/2023010600/20230106T00-00.nc`) and,
for the comparison curve, notebook 01's `era5_init_metrics.npz`.

In [ ]:
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from dateutil.relativedelta import relativedelta

from xichen.data import VARIABLES
from xichen.device import get_device
from xichen.forecast_eval import (
    eval_forecast, load_init_times, make_dacycle_init_loader, make_lr_loader,
)
from xichen import nwp

DATA_DIR = Path("/fs6/home/yangjh_data/project_data/xichen")
ERA5_DIR = Path("/fs6/home/yangjh_data/project_data/xichen/observation/ERA5")
CKPT_DIR = Path("/fs6/home/yangjh15/xichen/ckpt/xichen_1p0deg")
IC_DIR = Path("/fs6/home/yangjh15/xichen/XiChen_1p0deg_public/outputs/demo_dacycle/ic_6h")
OUTPUT_DIR = Path("/fs6/home/yangjh15/xichen/XiChen_1p0deg_public/outputs/da_init_forecast")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FORECAST_HOURS = 240
DT = 6
device = get_device("cuda", 0)

init_times = load_init_times("../configs/demo_init_times.json")
print("init times:", [t.isoformat() for t in init_times])
for t in init_times:
    p = nwp.analysis_ic_path(IC_DIR, t)
    assert Path(p).exists(), f"missing analysis {p} — run notebook 02 first"
print("all analyses present under", IC_DIR)

In [ ]:
# 10-day AR forecast from the DA analysis (init_loader reads ic_6h NetCDF).
config = {
    "era5_lr_dir": str(ERA5_DIR),
    "forecast_config": "../configs/xichen_forecast.json",
    "forecast_hours": FORECAST_HOURS,
    "dt": DT,
    "forecast_name": "xichen_dainit_demo",
    "device": device,
    "forecast_pair": "lr",
    "resolution_tag": "dainit_1p0deg",
    "output_resolution": "1p0",
    "init_times": init_times,
    "eval_batch_size": 1,
}
metrics_da = eval_forecast(
    make_lr_loader(str(ERA5_DIR)),
    str(CKPT_DIR / "xichen_state_forecast_ar15.ckpt"),
    config,
    str(OUTPUT_DIR),
    init_loader=make_dacycle_init_loader(str(IC_DIR)),
)

In [ ]:
# RMSE: DA-init forecast vs ERA5-init baseline (notebook 01), z-500 & t2m.
leads = np.arange(0, FORECAST_HOURS + 1, DT)
baseline = np.load("/fs6/home/yangjh15/xichen/XiChen_1p0deg_public/outputs/era5_init_forecast/era5_init_metrics.npz")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, v in zip(axes, ["z-500", "t2m"]):
    i = VARIABLES.index(v)
    ax.plot(leads, baseline["rmse"][i], marker="o", ms=3, label="ERA5-init (nb 01)")
    ax.plot(leads, metrics_da["rmse"][i], marker="s", ms=3, label="DA-init (this run)")
    ax.set_title(f"{v} RMSE")
    ax.set_xlabel("lead time [h]")
    ax.grid(alpha=0.3)
    ax.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "03_rmse_compare.png", dpi=300)
fig.savefig(OUTPUT_DIR / "03_rmse_compare.pdf", dpi=300)
plt.show()

In [ ]:
# DA-init forecast field vs ERA5 truth vs error, z-500 at t+24/120/240h.
Z500 = VARIABLES.index("z-500")
loader = make_lr_loader(str(ERA5_DIR))
INIT = init_times[0]

fig, axes = plt.subplots(3, 3, figsize=(15, 9), constrained_layout=True)
for row, lead in enumerate([24, 120, 240]):
    fc = nwp.load_field(nwp.field_path(OUTPUT_DIR / "forecast", INIT, lead))
    truth, _ = loader(INIT + relativedelta(hours=lead))
    fc_z, tr_z = fc[Z500], truth[Z500]
    err = fc_z - tr_z
    vmin, vmax = np.nanmin(tr_z), np.nanmax(tr_z)
    emax = np.nanmax(np.abs(err))
    for col, (data, title, cmap, lims) in enumerate([
        (tr_z, "ERA5 z-500", "viridis", (vmin, vmax)),
        (fc_z, f"DA-init z-500 t+{lead}h", "viridis", (vmin, vmax)),
        (err, f"error t+{lead}h", "RdBu_r", (-emax, emax)),
    ]):
        im = axes[row, col].imshow(data, cmap=cmap, vmin=lims[0], vmax=lims[1])
        axes[row, col].set_title(title)
        axes[row, col].axis("off")
        plt.colorbar(im, ax=axes[row, col], shrink=0.8)
fig.savefig(OUTPUT_DIR / "03_z500_fields.png", dpi=300)
fig.savefig(OUTPUT_DIR / "03_z500_fields.pdf", dpi=300)
plt.show()

This closes the loop: **ERA5 + observations → cascade DA → 10-day forecast**,
all from the released checkpoints. To reproduce the full one-year cycling of
the paper, see `configs/dacycle_oneyear.json` and README § Reproducing paper
results.